# 04 - Live event generator (append-only)
Appends the **next N minutes** of synthetic activity after the latest timestamp already in the raw tables, so the streaming Lakeflow pipeline picks it up incrementally. The 15-day base dataset (built once by `01_generate_synthetic_data`) is never overwritten.

Optional **scenario** injects a live incident for the demo:
* `playback_error_1002` - Android TV Error 1002 in BR/MX/AR after a player SDK deploy (quiet session drops; monitoring only alerts ~35 min later, incident declared ~45 min in)
* `subtitles_missing` - Spanish subtitles missing on a new episode (support-escalation only)
* `payment_gateway` - Adyen authorizations failing in DE/AT after a routing config change
* `login_outage` - APAC AUTH-503 login failures (monitoring catches it fast)

Appends to: `concurrent_streams`, `service_metrics`, `payment_transactions`, `support_tickets`, `ticket_accounts` (ticket -> subscriber), `social_posts`, `change_events`, `incidents`.

The last cell then runs the streaming Lakeflow pipeline `support_command_center_streaming` and refreshes the Lakebase serving tables, so one run of this notebook updates the app end to end.

In [0]:
dbutils.widgets.text("minutes", "60", "Minutes of activity to generate")
dbutils.widgets.dropdown("scenario", "none", ["none", "playback_error_1002", "subtitles_missing", "payment_gateway", "login_outage"], "Scenario to inject")
MINUTES = max(5, int(dbutils.widgets.get("minutes")))
SCENARIO = dbutils.widgets.get("scenario")
FQ = "classic_stable_l0m1dn_catalog.support_command_center"
print(MINUTES, SCENARIO)

In [0]:
import numpy as np, pandas as pd, random

BASE_START = pd.Timestamp("2026-10-13")
DEVICES = ["Android TV", "Fire TV", "Roku", "Apple TV", "iOS", "Android", "Web", "PlayStation", "Xbox"]
DSHARE = np.array([.14, .08, .10, .04, .16, .22, .14, .08, .04])
DOMAINS = ["Authentication", "Billing & Payments", "Playback & CDN", "Content Ingestion", "Subtitles & Localization", "Device Apps", "Search & Discovery"]
BASE_ERR = np.array([0.004, 0.010, 0.006, 0.002, 0.001, 0.003, 0.002]); BASE_LAT = np.array([180, 320, 140, 900, 60, 210, 120])
REQ_MULT = np.array([0.08, 0.01, 1.0, 0.002, 0.3, 0.2, 0.15])
SHOWS = ["Blade of the Crimson Moon", "Spirit Ronin", "Iron Samurai Gaiden", "Moonlit Idol Stage", "Mecha Academy Zero", "Isekai Cafe Chronicles", "Shadow Guild Hunters", "Ocean Spirit Diaries"]
PRICE = {"Stripe": 9.99, "Adyen": 10.49, "PayPal": 9.99, "Apple IAP": 11.99, "Google Play": 10.99}
EU_ADYEN = {"DE", "AT", "FR", "IT", "ES", "GB"}
CATS = {  # category: (contact rate per 10k streams per 5 min, form category, [(subject, body)])
 "playback_buffering": (0.10, "Playback", [("Video keeps buffering", "{show} keeps buffering every few minutes on my {device}. My internet is fine for other apps.")]),
 "playback_error_code": (0.04, "Playback", [("Error {code} when playing", "I get Error {code} when trying to start {show} on {device}.")]),
 "login_issue": (0.08, "Account & Login", [("Can't sign in", "I can't log in to my account on {device}, it says something went wrong.")]),
 "password_reset": (0.05, "Account & Login", [("Password reset email not received", "I requested a password reset but never got the email.")]),
 "payment_failed": (0.04, "Billing", [("Payment failed", "My renewal payment failed but my card is fine.")]),
 "billing_question": (0.05, "Billing", [("Charged twice", "I think I was charged twice this month for my membership.")]),
 "cancellation_request": (0.03, "Billing", [("Cancel my membership", "Please cancel my subscription before the next billing date.")]),
 "subtitles_missing": (0.015, "Subtitles & Audio", [("Subtitles not available", "Subtitles are not showing for {show} episode {ep}.")]),
 "subtitles_sync": (0.015, "Subtitles & Audio", [("Subtitles out of sync", "Subtitles are a few seconds late on {show} episode {ep}.")]),
 "dub_quality": (0.01, "Subtitles & Audio", [("Dub audio issue", "The English dub audio cuts out in {show} episode {ep}.")]),
 "episode_unavailable": (0.02, "Content", [("Episode not showing", "Episode {ep} of {show} is not available in my region yet?")]),
 "app_crash": (0.05, "App Issue", [("App crashes", "The app crashes when I open my watchlist on {device}.")]),
 "search_issue": (0.02, "App Issue", [("Search not working", "Search doesn't find {show} even though it's on the site.")]),
}


def cd(h, c):
    x = np.abs(h - c) % 24
    return np.minimum(x, 24 - x)


def generate(minutes, scenario):
    reg = spark.table(f"{FQ}.regions").toPandas()
    last = pd.Timestamp(spark.sql(f"SELECT max(ts) FROM {FQ}.concurrent_streams").first()[0])
    start = last + pd.Timedelta(minutes=5)
    T = max(1, minutes // 5)
    TS = pd.date_range(start, periods=T, freq="5min")
    rng = np.random.default_rng(int(start.timestamp()) // 300)
    random.seed(int(start.timestamp()))
    C = len(reg); cidx = {c: i for i, c in enumerate(reg.country_code)}
    share = np.tile(DSHARE, (C, 1))
    for c in ["BR", "MX", "AR"]:
        if c in cidx: share[cidx[c], 0] = 0.22
    share = share / share.sum(1, keepdims=True)

    # ---- demand model (same shape as the base data)
    mod = TS.hour.values * 60 + TS.minute.values
    day_idx = ((TS - BASE_START).days).values
    off = reg.utc_offset.values
    lh = mod[:, None] / 60 + off[None, :]
    lday = np.floor(lh / 24) + day_idx[:, None]; lh = lh % 24
    wkd = (BASE_START.weekday() + lday) % 7
    diurnal = 0.08 + 0.92 * np.exp(-cd(lh, 21) ** 2 / (2 * 2.8 ** 2)) + 0.15 * np.exp(-cd(lh, 13) ** 2 / (2 * 2.2 ** 2))
    base = reg.subscribers.values[None, :] * 0.075 * diurnal * np.where(wkd >= 5, 1.15, 1.0) * (1 + 0.002 * day_idx[:, None])
    streams_nf = base[:, :, None] * share[None, :, :] * rng.lognormal(0, 0.03, (T, C, len(DEVICES)))
    factor = np.ones_like(streams_nf)
    err_mult = np.ones((T, C, len(DOMAINS)))
    gw_fail = {}
    onset = min(2, T - 1)                      # incident starts 10 minutes into the window
    w = np.zeros(T); w[onset:] = 1.0
    late = np.zeros(T); late[min(T - 1, onset + 7):] = 1.0   # monitoring threshold crossed ~35 min after onset
    t_on = TS[onset]
    inc_rows, chg_rows, extra_tickets, extra_posts = [], [], [], []
    gi = {g: i for i, g in enumerate(DOMAINS)}
    di = {d: i for i, d in enumerate(DEVICES)}

    def incident(title, domain, sev, countries, rgs, alert_after, declare_after, dur, detected_by, cause):
        fa = t_on + pd.Timedelta(minutes=alert_after) if alert_after is not None else pd.NaT
        da = t_on + pd.Timedelta(minutes=declare_after)
        inc_rows.append(dict(incident_id=f"INC-{t_on:%m%d-%H%M}", title=title, domain=domain, severity=sev, countries=",".join(countries),
                             region_groups=rgs, first_alert_at=fa, declared_at=da, resolved_at=da + pd.Timedelta(minutes=dur),
                             detected_by=detected_by, root_cause=cause))

    def change(service, domain, ctype, version, desc, scope):
        chg_rows.append(dict(change_ts=t_on - pd.Timedelta(minutes=6), service=service, domain=domain, change_type=ctype, version=version,
                             description=desc, scope=scope, outcome="rolled_back"))

    if scenario == "playback_error_1002":
        for c, depth in [("BR", 0.45), ("MX", 0.2), ("AR", 0.2)]:
            factor[:, cidx[c], di["Android TV"]] *= (1 - depth * w)
            err_mult[:, cidx[c], gi["Playback & CDN"]] *= (1 + 0.4 * w + 6 * late)
        change("player-sdk-androidtv", "Playback & CDN", "deployment", "v2.4.0", "Android TV player SDK: new adaptive connection pooling for LATAM edges", "LATAM")
        incident("Android TV playback failures in LATAM (Error 1002)", "Playback & CDN", "SEV1", ["BR", "MX", "AR"], "LATAM", 35, 45, 60,
                 "Monitoring (delayed)", "Android TV player SDK v2.4.0 connection pooling silently dropping sessions")
    elif scenario == "subtitles_missing":
        show, ep = random.choice(SHOWS[2:]), random.randint(4, 12)
        for c in ["MX", "ES", "AR", "CO", "CL"]:
            err_mult[:, cidx[c], gi["Subtitles & Localization"]] *= (1 + 0.8 * w)
        change("content-pipeline", "Content Ingestion", "content_publish", f"job-{random.randint(56000, 59999)}",
               f"Publish {show} E{ep} + subtitle packs (es-419 pack failed validation silently)", "Global")
        incident(f"Spanish subtitles missing - {show} E{ep}", "Subtitles & Localization", "SEV3", ["MX", "ES", "AR", "CO", "CL"], "LATAM,EMEA",
                 None, 70, 45, "Support escalation", "es-419 subtitle pack failed validation silently during publish")
        extra_tickets.append(("subs", ["MX", "ES", "AR", "CO", "CL"], None, 0.00025, show, ep))
    elif scenario == "payment_gateway":
        for c in ["DE", "AT"]:
            gw_fail[(c, "Adyen")] = 0.06 * w + 0.21 * late
            err_mult[:, cidx[c], gi["Billing & Payments"]] *= (1 + 1.0 * w + 6 * late)
        change("payments-router", "Billing & Payments", "config_change", f"cfg-{random.randint(9100, 9999)}", "Shift Adyen EU traffic to new regional endpoint", "EMEA")
        incident("Adyen payment failures in DE/AT", "Billing & Payments", "SEV2", ["DE", "AT"], "EMEA", 40, 45, 60, "Monitoring",
                 "Adyen EU endpoint change misrouted authorizations")
    elif scenario == "login_outage":
        cs = ["JP", "IN", "AU", "PH", "ID", "KR"]
        for c in cs:
            factor[:, cidx[c], :] *= (1 - 0.15 * w)[:, None]
            err_mult[:, cidx[c], gi["Authentication"]] *= (1 + 29 * w)
        change("auth-service", "Authentication", "config_change", f"cfg-{random.randint(8900, 8999)}", "Token cache TTL change for APAC edge", "APAC")
        incident("APAC login failures (AUTH-503)", "Authentication", "SEV1", cs, "APAC", 4, 6, 45, "Monitoring", "Token cache TTL config change at APAC edge")

    streams = np.round(streams_nf * factor).astype(np.int64)
    lost = np.maximum(0, streams_nf * (1 - factor))
    ii, cc, dd = np.meshgrid(np.arange(T), np.arange(C), np.arange(len(DEVICES)), indexing="ij")
    err = np.clip(BASE_ERR[None, None, :] * err_mult * rng.lognormal(0, 0.12, (T, C, len(DOMAINS))), 0, 0.95)
    cs_df = pd.DataFrame({"ts": TS.values[ii.ravel()], "country_code": reg.country_code.values[cc.ravel()], "device": np.array(DEVICES)[dd.ravel()],
                          "concurrent_streams": streams.ravel(),
                          "playback_start_failures": np.round(streams.ravel() * np.repeat(err[:, :, 2], len(DEVICES), axis=1).reshape(-1) * 0.2 + lost.ravel() * 0.05).astype(np.int64)})
    lat = BASE_LAT[None, None, :] * (1 + 0.5 * (err_mult - 1) / np.maximum(err_mult, 1)) * rng.lognormal(0, 0.06, err.shape)
    req = (streams.sum(2)[:, :, None] * REQ_MULT[None, None, :]).astype(np.int64) + 10
    ii, cc, gg = np.meshgrid(np.arange(T), np.arange(C), np.arange(len(DOMAINS)), indexing="ij")
    sm_df = pd.DataFrame({"ts": TS.values[ii.ravel()], "country_code": reg.country_code.values[cc.ravel()], "domain": np.array(DOMAINS)[gg.ravel()],
                          "request_count": req.ravel(), "error_rate": np.round(err.ravel(), 5), "p95_latency_ms": np.round(lat.ravel(), 1)})

    # ---- payments
    renew = 3 * (reg.subscribers.values * 0.8 / 30 / 288)[None, :] * (0.3 + 1.4 * np.exp(-cd(lh, 14) ** 2 / (2 * 4 ** 2)))
    pay_rows, pay_fail = [], np.zeros((T, C))
    for c in reg.country_code:
        ci = cidx[c]
        gws = {"Stripe": .35, "PayPal": .2, "Apple IAP": .2, "Google Play": .25} if c not in EU_ADYEN else {"Stripe": .15, "Adyen": .3, "PayPal": .2, "Apple IAP": .15, "Google Play": .2}
        for gw, sh in gws.items():
            att = rng.poisson(renew[:, ci] * sh)
            gf = rng.binomial(att, np.clip(0.005 + gw_fail.get((c, gw), np.zeros(T)), 0, 1))
            cf = rng.binomial(att - gf, 0.025 if gw in ("Stripe", "Adyen") else 0.012)
            pay_fail[:, ci] += gf
            pay_rows.append(pd.DataFrame({"ts": TS, "country_code": c, "gateway": gw, "attempts": att, "successes": att - gf - cf,
                                          "customer_declines": cf, "gateway_errors": gf, "avg_price_usd": PRICE[gw]}))
    pay_df = pd.concat(pay_rows, ignore_index=True)

    # ---- tickets: baseline contact rates + scenario surges (text matches the silver signature rules)
    lang_of = dict(zip(reg.country_code, reg.primary_language))
    rows = []
    def add_ticket(t, c, d, cat, subj, body, scen=False):
        rows.append((TS[t] + pd.Timedelta(seconds=int(rng.integers(0, 300))), c, DEVICES[d], cat, subj, body, scen))
    for k, (rate, form, tpls) in CATS.items():
        n = rng.poisson((streams / 1e4) * rate)
        for t, c, d in zip(*np.nonzero(n)):
            for _ in range(n[t, c, d]):
                s, b = random.choice(tpls)
                f = dict(show=random.choice(SHOWS), ep=random.randint(1, 12), device=DEVICES[d], code=random.choice(["2001", "3004", "4011", "5003"]))
                add_ticket(t, reg.country_code[c], d, k, s.format(**f), b.format(**f))
    if scenario == "playback_error_1002":
        pt = {"Portuguese": ("Erro 1002 na Android TV", "Toda vez que tento assistir {show} aparece Erro 1002 na minha Android TV."),
              "Spanish": ("Error 1002 en Android TV", "No puedo ver {show}, me sale error 1002 en mi Android TV.")}
        for c in ["BR", "MX", "AR"]:
            lam = lost[:, cidx[c], di["Android TV"]] * 0.0025 + (4 if c == "BR" else 0.8) * w
            for t, n in enumerate(rng.poisson(lam)):
                for _ in range(n):
                    s, b = pt[lang_of[c]] if random.random() < 0.55 else ("Error 1002 on Android TV", "I keep getting Error 1002 when I try to play {show} on my Android TV.")
                    sh = random.choice(SHOWS); add_ticket(t, c, di["Android TV"], "playback_error_code", s, b.format(show=sh), True)
    for kind, cs, _, rate, show, ep in extra_tickets:
        for c in cs:
            lam = streams[:, cidx[c], :] * rate * w[:, None]
            n = rng.poisson(lam)
            for t, d in zip(*np.nonzero(n)):
                for _ in range(n[t, d]):
                    if random.random() < 0.5:
                        s, b = "Faltan subtítulos en español", f"El episodio {ep} de {show} no tiene subtítulos en español."
                    else:
                        s, b = f"Spanish subtitles missing - {show} episode {ep}", f"Episode {ep} of {show} has no Spanish subtitles."
                    add_ticket(t, c, d, "subtitles_missing", s, b, True)
    if scenario == "payment_gateway":
        for c in ["DE", "AT"]:
            lam = pay_fail[:, cidx[c]] * 0.8 + streams[:, cidx[c], :].sum(1) * 0.00015 * w
            for t, n in enumerate(rng.poisson(lam)):
                for _ in range(n):
                    s, b = ("Zahlung fehlgeschlagen", "Meine Kreditkartenzahlung wird abgelehnt (payment could not be processed).") if random.random() < 0.5 else \
                           ("Payment declined renewing subscription", "My card was declined when renewing Mega Fan. Message: payment could not be processed.")
                    add_ticket(t, c, int(rng.integers(0, len(DEVICES))), "payment_failed", s, b, True)
    if scenario == "login_outage":
        for c in ["JP", "IN", "AU", "PH", "ID", "KR"]:
            lam = streams[:, cidx[c], :] * 0.0012 * w[:, None]
            n = rng.poisson(lam)
            for t, d in zip(*np.nonzero(n)):
                for _ in range(n[t, d]):
                    add_ticket(t, c, d, "login_issue", "Can't log in - AUTH-503", f"Login spinner then 'Something went wrong (AUTH-503)' on {DEVICES[d]}.", True)
    tk = pd.DataFrame(rows, columns=["created_at", "country_code", "device", "cat", "subject", "description", "scen"]).sort_values("created_at").reset_index(drop=True)
    n = len(tk)
    max_id = spark.sql(f"SELECT max(CAST(substr(ticket_id, 4) AS BIGINT)) FROM {FQ}.support_tickets").first()[0] or 4800000
    tk["ticket_id"] = [f"CR-{max_id + 1 + i}" for i in range(n)]
    tk["language"] = [lang_of[c] for c in tk.country_code]
    tk["channel"] = rng.choice(["Chat", "Email", "In-App", "Social"], n, p=[.45, .25, .2, .1])
    tk["subscriber_tier"] = rng.choice(["Free", "Fan", "Mega Fan", "Ultimate Fan"], n, p=[.3, .35, .28, .07])
    r2 = rng.random(n)
    tk["form_category"] = np.where(r2 < 0.7, [CATS[c][1] for c in tk.cat], "Other")
    surge = scenario != "none"
    frt = np.array([{"Chat": 3, "Email": 90, "In-App": 30, "Social": 45}[c] for c in tk.channel]) * rng.lognormal(0, 0.5, n) * (1.8 if surge else 1.0)
    tk["first_response_minutes"] = np.round(frt, 1)
    tk["sla_target_minutes"] = [{"Free": 240, "Fan": 120, "Mega Fan": 60, "Ultimate Fan": 30}[x] for x in tk.subscriber_tier]
    tk["resolved_at"] = tk.created_at + pd.to_timedelta(frt + rng.lognormal(0.4, 0.8, n) * 60, unit="m")
    tk["csat_score"] = np.where(rng.random(n) < 0.35, np.clip(np.round(rng.normal(4.3 - (0.7 if surge else 0), 0.8, n)), 1, 5), np.nan)
    tk["reopened"] = rng.random(n) < (0.08 if surge else 0.04)
    tickets = tk[["ticket_id", "created_at", "country_code", "language", "device", "channel", "subscriber_tier", "form_category", "subject", "description",
                  "first_response_minutes", "sla_target_minutes", "resolved_at", "csat_score", "reopened"]]

    # ---- link each new ticket to a subscriber account of the same country + tier.
    #      ~35% of incident tickets come from subscribers who already reported the incident in this batch (repeat contacts).
    sb = spark.table(f"{FQ}.subscribers").select("subscriber_id", "country_code", "tier").toPandas()
    pools = sb.groupby(["country_code", "tier"]).subscriber_id.apply(np.array).to_dict()
    reporters, sub_ids = {}, []
    for r in tk.itertuples():
        key = (r.country_code, r.subscriber_tier)
        if r.scen and reporters.get(key) and rng.random() < 0.35:
            sid = random.choice(reporters[key])
        else:
            sid = str(rng.choice(pools[key]))
        if r.scen:
            reporters.setdefault(key, []).append(sid)
        sub_ids.append(sid)
    ticket_accounts = pd.DataFrame({"ticket_id": tk.ticket_id, "subscriber_id": sub_ids, "linked_at": tk.created_at})

    # ---- social posts
    hype = ["{show} episode {ep} was INSANE this week", "Can't believe that ending of {show}!", "{show} is the best show of the season"]
    neg = {"playback_error_1002": ["Fanimation down on Android TV? Error 1002 every episode #FanimationDown", "Erro 1002 na Fanimation de novo, Android TV não funciona"],
           "subtitles_missing": ["Fanimation olvidó los subtítulos en español otra vez", "No Spanish subs on the new episode, really @Fanimation?"],
           "payment_gateway": ["Fanimation Zahlung wird abgelehnt, jemand anderes auch?", "Fanimation keeps declining my card on renewal"],
           "login_outage": ["Fanimation logged me out and AUTH-503 won't let me back in", "Is Fanimation login broken in Asia right now?"]}
    ctry = {"playback_error_1002": ["BR", "MX", "AR"], "subtitles_missing": ["MX", "ES", "AR"], "payment_gateway": ["DE", "AT"], "login_outage": ["JP", "IN", "AU", "PH"]}
    posts = []
    for t in range(T):
        for _ in range(rng.poisson(0.6)):
            posts.append((TS[t], random.choice(hype).format(show=random.choice(SHOWS), ep=random.randint(1, 12)), random.choice(list(reg.country_code))))
        if scenario in neg and w[t] and t > onset:
            for _ in range(rng.poisson(2.5)):
                posts.append((TS[t], random.choice(neg[scenario]), random.choice(ctry[scenario])))
    soc = pd.DataFrame(posts, columns=["posted_at", "post_text", "country_code"])
    max_sp = spark.sql(f"SELECT max(CAST(substr(post_id, 4) AS BIGINT)) FROM {FQ}.social_posts").first()[0] or 100000
    soc["posted_at"] = soc.posted_at + pd.to_timedelta(rng.integers(0, 300, len(soc)), unit="s")
    soc = soc.sort_values("posted_at").reset_index(drop=True)
    soc["post_id"] = [f"SP-{max_sp + 1 + i}" for i in range(len(soc))]
    soc["platform"] = rng.choice(["Reddit", "X"], len(soc))
    soc["author_handle"] = ["@" + random.choice(["otaku", "animefan", "weeb", "senpai", "kitsune"]) + str(random.randint(10, 9999)) for _ in range(len(soc))]
    soc["engagement"] = rng.negative_binomial(2, 0.05, len(soc)).astype(np.int64)
    soc = soc[["post_id", "posted_at", "platform", "author_handle", "country_code", "post_text", "engagement"]]

    # ---- change events + incidents
    max_chg = spark.sql(f"SELECT max(CAST(substr(change_id, 5) AS BIGINT)) FROM {FQ}.change_events").first()[0] or 20000
    svc = [("search-api", "Search & Discovery"), ("app-ios", "Device Apps"), ("billing-api", "Billing & Payments"), ("playback-service", "Playback & CDN")]
    for _ in range(rng.poisson(max(0.3, minutes / 240))):
        s, d = random.choice(svc)
        chg_rows.append(dict(change_ts=TS[int(rng.integers(0, T))], service=s, domain=d, change_type="deployment",
                             version=f"v{random.randint(1, 4)}.{random.randint(0, 20)}.{random.randint(0, 9)}", description="Routine release", scope="Global", outcome="success"))
    chg = pd.DataFrame(chg_rows, columns=["change_ts", "service", "domain", "change_type", "version", "description", "scope", "outcome"])
    chg.insert(0, "change_id", [f"CHG-{max_chg + 1 + i}" for i in range(len(chg))])
    inc = pd.DataFrame(inc_rows, columns=["incident_id", "title", "domain", "severity", "countries", "region_groups", "first_alert_at", "declared_at", "resolved_at", "detected_by", "root_cause"])
    return {"concurrent_streams": cs_df, "service_metrics": sm_df, "payment_transactions": pay_df, "support_tickets": tickets,
            "social_posts": soc, "change_events": chg, "incidents": inc, "ticket_accounts": ticket_accounts}, TS

In [0]:
batch, TS = generate(MINUTES, SCENARIO)
print(f"Window: {TS[0]} -> {TS[-1] + pd.Timedelta(minutes=5)} UTC  |  scenario: {SCENARIO}")
for name, pdf in batch.items():
    print(f"{name:22s} {len(pdf):>8,} rows")
display(batch["support_tickets"].head(5))

In [0]:
from pyspark.sql import functions as F

# Append with the exact target schema so the streaming pipeline picks up the new rows incrementally
for name, pdf in batch.items():
    if pdf.empty:
        continue
    target = spark.table(f"{FQ}.{name}").schema
    sdf = spark.createDataFrame(pdf.astype(object).where(pd.notnull(pdf), None)) if name == "incidents" else spark.createDataFrame(pdf)
    sdf = sdf.select([F.col(f.name).cast(f.dataType).alias(f.name) for f in target])
    sdf.write.mode("append").saveAsTable(f"{FQ}.{name}")
    print(f"appended {len(pdf):,} rows to {name}")

In [0]:
import subprocess, sys, time
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "databricks-sdk>=0.118.0", "psycopg[binary]"])
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()
PIPELINE_ID = "078c3ff3-427b-4c4b-846a-61781f1f524b"   # support_command_center_streaming
SDP = "classic_stable_l0m1dn_catalog.support_command_center_sdp"
SYNCED = ["region_heatmap_5m", "service_health_region_5m", "frustration_index_5m", "ticket_anomalies", "incident_scorecard", "signature_trends_15m",
          "payment_health_5m", "social_posts", "exec_daily_kpis", "change_events", "incidents", "subscriber_360", "subscriber_ticket_history",
          "churn_risk_by_issue", "customer_outcomes_daily"]

def wait_update(pipeline_id, update_id, label):
    while True:
        st = w.pipelines.get_update(pipeline_id=pipeline_id, update_id=update_id).update.state.value
        if st in ("COMPLETED", "FAILED", "CANCELED"):
            print(f"{label}: {st}"); return st
        time.sleep(10)

# 1) incremental update of the streaming pipeline (only the newly appended rows are processed)
t0 = time.time()
up = w.pipelines.start_update(pipeline_id=PIPELINE_ID)
assert wait_update(PIPELINE_ID, up.update_id, "support_command_center_streaming") == "COMPLETED"

# 2) refresh the Lakebase synced tables (snapshot) that serve the app
runs = {}
for n in SYNCED:
    pid = w.postgres.get_synced_table(name=f"synced_tables/{SDP}.lb_{n}").status.pipeline_id
    runs[n] = (pid, w.pipelines.start_update(pipeline_id=pid).update_id)
for n, (pid, uid) in runs.items():
    wait_update(pid, uid, f"lb_{n}")

# 3) re-apply app lookup indexes (idempotent)
import psycopg
ep = w.postgres.get_endpoint(name="projects/support-command-center/branches/production/endpoints/primary")
with psycopg.connect(host=ep.status.hosts.host, dbname="databricks_postgres", user=w.current_user.me().user_name,
                     password=w.postgres.generate_database_credential(endpoint=ep.name).token, sslmode="require") as c:
    for s in ["CREATE INDEX IF NOT EXISTS ix_sth_sub ON support_command_center_sdp.lb_subscriber_ticket_history (subscriber_id, created_at)",
              "CREATE INDEX IF NOT EXISTS ix_s360_risk ON support_command_center_sdp.lb_subscriber_360 (churn_risk_band, churn_risk_score DESC)",
              "CREATE INDEX IF NOT EXISTS ix_s360_email ON support_command_center_sdp.lb_subscriber_360 (email)"]:
        c.execute(s)
    c.commit()
print(f"Done in {time.time() - t0:.0f}s - the app now shows the new data.")